# 04. 対数 — Python で「桁」 を体感する

**このノートブックの内容**:
1. 対数の基本: `math.log10`, `math.log2`, `math.log`
2. 桁を圧縮する道具としての対数 (地震・音・pH)
3. 対数グラフで指数的成長を直線化する
4. 機械学習で現れる対数 (クロスエントロピー)
5. 対数の 3 法則を Python で確認

各セルを **Shift+Enter** で実行してください。

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (04_logarithm.md)](../04_logarithm.md)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, FloatSlider

%matplotlib inline

## 1. 対数の基本 — 「何回かけたら?」 を計算する

$\log_a(x) = y \quad \Leftrightarrow \quad a^y = x$

「**a を何回かけたら x になるか**」 が y。

In [ ]:
# 10 を何回かけたら 100 になる? → 2 回
print(f"log10(100) = {math.log10(100)}")

# 10 を何回かけたら 1,000,000 になる? → 6 回
print(f"log10(1,000,000) = {math.log10(1_000_000)}")

# 2 を何回かけたら 1024 になる? → 10 回 (1KB = 2^10 B の正体)
print(f"log2(1024) = {math.log2(1024)}")

# 自然対数 (底 e ≈ 2.718)
print(f"ln(e) = {math.log(math.e)}")

# 任意の底
print(f"log_2(8) = {math.log(8, 2)}")

## 2. 桁の圧縮 — 地震マグニチュードと音のデシベル

マグニチュードが 1 上がるとエネルギーは約 **32 倍**、2 上がると約 **1000 倍**。
桁が違いすぎる量を「人間に分かる数字」 に圧縮するのが対数の威力。

In [ ]:
# 地震マグニチュードとエネルギー比 (M5 を 1 とした相対値)
magnitudes = np.array([5, 6, 7, 8, 9])
# エネルギー = 32^(M - 5) というモデル
energy_ratio = 32 ** (magnitudes - 5)

for m, e in zip(magnitudes, energy_ratio):
    print(f"M{m}: エネルギー比 ≈ {e:>15,.0f} 倍")

print("\n→ M5 → M9 で 100 万倍超! 数字 4 違うだけでこれだけ違う")

In [ ]:
# 音のデシベル (dB) → 音圧の倍率
# 10 dB の差 = 10 倍、20 dB = 100 倍、30 dB = 1000 倍
db_values = np.array([0, 30, 60, 80, 100, 120])
labels = ["無音", "ささやき声", "会話", "掃除機", "地下鉄車内", "ジェット機"]
intensity = 10 ** (db_values / 10)

for db, lbl, i in zip(db_values, labels, intensity):
    print(f"{db:>3} dB ({lbl:<10}): 音の強さ {i:>15,.0f}")

print("\n→ ささやき (30dB) と地下鉄 (100dB) で音の強さは 1000 万倍!")

## 3. 対数グラフで指数的成長を直線化する

「年 5% 成長」 を 100 年続けると、グラフは指数曲線になり、後半が急激すぎて初期が見えない。
**y 軸を対数スケール** にすると、これが**直線**になり比較しやすくなる。

In [ ]:
years = np.arange(0, 101)
value_5pct = 100 * (1.05 ** years)
value_3pct = 100 * (1.03 ** years)
value_10pct = 100 * (1.10 ** years)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 通常スケール: 後半に潰される
axes[0].plot(years, value_3pct, label="年 3% 成長")
axes[0].plot(years, value_5pct, label="年 5% 成長")
axes[0].plot(years, value_10pct, label="年 10% 成長")
axes[0].set_title("通常スケール (初期がほぼ見えない)")
axes[0].set_xlabel("年")
axes[0].set_ylabel("資産価値")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 対数スケール: 直線になり比較が容易
axes[1].semilogy(years, value_3pct, label="年 3% 成長")
axes[1].semilogy(years, value_5pct, label="年 5% 成長")
axes[1].semilogy(years, value_10pct, label="年 10% 成長")
axes[1].set_title("対数スケール (傾きが成長率を表す)")
axes[1].set_xlabel("年")
axes[1].set_ylabel("資産価値 (log)")
axes[1].legend()
axes[1].grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.show()

## 4. 機械学習で現れる対数 — クロスエントロピー

分類問題の損失関数 **クロスエントロピー** は対数を使います:

$$L = -\sum_i y_i \log(\hat{y}_i)$$

**確信があるほど損失は小さく、自信がないほど損失は爆発的に大きく** なる性質を、対数で実現しています。

In [ ]:
# 「正解」 に対するモデルの予測確率を p としたとき、損失 = -log(p)
p_values = np.linspace(0.01, 1.0, 100)
loss = -np.log(p_values)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(p_values, loss, linewidth=2)
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_title("クロスエントロピー損失: $L = -\\log(p)$")
ax.set_xlabel("モデルの予測確率 p (正解クラス)")
ax.set_ylabel("損失 L")
ax.grid(True, alpha=0.3)

# 注釈
ax.annotate("確信あり (p≈1)\n→ 損失ほぼ 0", xy=(0.95, 0.05), xytext=(0.55, 1.5),
            arrowprops=dict(arrowstyle="->", color="green"))
ax.annotate("自信なし (p小)\n→ 損失爆発", xy=(0.05, -np.log(0.05)), xytext=(0.2, 3.5),
            arrowprops=dict(arrowstyle="->", color="red"))

plt.tight_layout()
plt.show()

print(f"p=0.99 のとき損失 = {-np.log(0.99):.4f}")
print(f"p=0.50 のとき損失 = {-np.log(0.50):.4f}")
print(f"p=0.10 のとき損失 = {-np.log(0.10):.4f}")
print(f"p=0.01 のとき損失 = {-np.log(0.01):.4f}")

## 5. 対数の 3 法則を Python で確認

| 法則 | 式 |
|---|---|
| 掛け算 → 足し算 | $\log(a \cdot b) = \log(a) + \log(b)$ |
| 割り算 → 引き算 | $\log(a / b) = \log(a) - \log(b)$ |
| べき乗 → 掛け算 | $\log(a^n) = n \cdot \log(a)$ |

「コンピュータが無い時代に複雑な計算ができた」 のは、対数表 + 足し算 で掛け算を済ませたから。

In [ ]:
a, b, n = 12.3, 45.6, 7

# 法則 1: log(a*b) == log(a) + log(b)
left  = math.log10(a * b)
right = math.log10(a) + math.log10(b)
print(f"log({a}*{b}) = {left:.6f}")
print(f"log({a}) + log({b}) = {right:.6f}")
print(f"差: {abs(left - right):.2e}  (誤差はほぼゼロ)\n")

# 法則 2: log(a/b) == log(a) - log(b)
left  = math.log10(a / b)
right = math.log10(a) - math.log10(b)
print(f"log({a}/{b}) = {left:.6f}")
print(f"log({a}) - log({b}) = {right:.6f}")
print(f"差: {abs(left - right):.2e}\n")

# 法則 3: log(a^n) == n*log(a)
left  = math.log10(a ** n)
right = n * math.log10(a)
print(f"log({a}^{n}) = {left:.6f}")
print(f"{n} * log({a}) = {right:.6f}")
print(f"差: {abs(left - right):.2e}")

## 6. 対話的に対数を体感 — スライダーで底を変える

底 $a$ を変えると $\log_a(x)$ のカーブはどう変わる?

In [ ]:
def plot_log(base: float = 10.0) -> None:
    x = np.linspace(0.1, 100, 500)
    y = np.log(x) / np.log(base)  # 任意底の対数
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(x, y, linewidth=2)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(1, color="gray", linewidth=0.5)
    ax.set_title(f"$y = \\log_{{{base:.1f}}}(x)$")
    ax.set_xlabel("x")
    ax.set_ylabel(f"log_{{{base:.1f}}}(x)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_log, base=FloatSlider(min=2.0, max=20.0, step=0.5, value=10.0, description="底 a"));

## ⚠️ よくあるエラー — log の定義域

対数は「**正の実数**」 でのみ定義されます。0 や負数を入れると Python は素直にエラーで教えてくれます。

In [ ]:
import math

for val in [10, 1, 0, -1]:
    try:
        print(f'math.log({val}) = {math.log(val):.4f}')
    except ValueError as e:
        print(f'math.log({val}) → ❌ {type(e).__name__}: {e}')

# → 数学的事実 (log の定義域 = 正の実数) が、Python の挙動として現れる
# エラー読解の詳細: start_here/00_pet_terminal/columns/05_reading_errors.md

---

## 📚 まとめ

- **対数 = 桁の道具**: 巨大な数を人間サイズに圧縮する
- **指数的成長を直線化** する対数スケール (`semilogy`) はデータ可視化の基本
- **機械学習のクロスエントロピー** は対数を使った損失関数 — 自信のない予測ほどキツくペナルティ
- **対数の 3 法則** (積→和 / 商→差 / 冪→積) は、ニューラルネット内部の式変形でも頻出

次は数学記号と論理 → [`00_notation/`](../../00_notation/README.md) に進みましょう。

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [`03_trigonometry.ipynb`](03_trigonometry.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [次の章: 00_notation](../../00_notation/README.md) |